# 03 — Outliers

Companion to [`../../data_cleaning/outliers.md`](../../data_cleaning/outliers.md).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(0)
n = 1000
income = rng.lognormal(10.5, 0.7, n)
# inject 1% extreme outliers
income[rng.choice(n, 10, replace=False)] *= 50
df = pd.DataFrame({'income': income})
df.describe()

## 1. Visual inspection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
sns.histplot(df['income'], bins=50, ax=axes[0]); axes[0].set_title('Linear')
sns.histplot(np.log1p(df['income']), bins=50, ax=axes[1]); axes[1].set_title('log1p')
sns.ecdfplot(df['income'], ax=axes[2]); axes[2].set_xscale('log'); axes[2].set_title('ECDF (log x)')
plt.tight_layout(); plt.show()

## 2. IQR rule

In [ ]:
q1, q3 = df['income'].quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
df['iqr_outlier'] = (df['income'] < lo) | (df['income'] > hi)
df['iqr_outlier'].sum()

## 3. Modified z-score (robust)

In [ ]:
med = df['income'].median()
mad = (df['income'] - med).abs().median()
df['mz'] = 0.6745 * (df['income'] - med) / mad
df['mz_outlier'] = df['mz'].abs() > 3.5
df['mz_outlier'].sum()

## 4. Isolation Forest (multivariate-ready)

In [ ]:
iso = IsolationForest(contamination=0.01, random_state=0).fit(df[['income']])
df['iso_outlier'] = iso.predict(df[['income']]) == -1
df['iso_outlier'].sum()

## 5. Comparison

In [ ]:
df[['iqr_outlier', 'mz_outlier', 'iso_outlier']].sum()

## Response — what to do

Recall from the chapter: never drop outliers silently. Options:

- Log-transform first (often makes the "outliers" disappear).
- Winsorize (cap at the 1st / 99th percentile).
- Use a robust model (Huber regression, tree models).
- If genuine errors: drop with an audit log.